### 请求 Base URL
https://apihub.agnes-ai.com/v1

## 短视频剧本生成 - 基于小说转脚本

### (1) 加载环境变量

In [ ]:
import os

api_key = os.getenv('AGNES_API_KEY')
if not api_key:
    print("⚠️ 未找到 AGNES_API_KEY 环境变量，请设置后再运行")
else:
    print("✅ AGNES_API_KEY 已加载")

### (2) 加载短视频剧本技能规则

In [ ]:
# 读取 Skills/script.md 中的剧本规则
# notebook 可能从 test/ 目录运行，也可能从项目根目录运行
_cwd = os.getcwd()
# 获取 notebook 所在目录（假设是 test/）
_notebook_dir = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else _cwd
# 项目根目录：从 test/ 向上一级，或直接就是当前目录
_project_root = os.path.dirname(_cwd) if os.path.basename(_cwd) == "test" else _cwd

_possible_paths = [
    os.path.join(_project_root, "Skills", "script.md"),      # 从项目根目录
    os.path.join(_cwd, "Skills", "script.md"),              # 从当前目录
    os.path.join(_cwd, "..", "Skills", "script.md"),       # 从 test/ 向上一级
]

script_skills_path = None
for _p in _possible_paths:
    _p_normalized = os.path.normpath(_p)
    if os.path.exists(_p_normalized):
        script_skills_path = os.path.abspath(_p_normalized)
        break

if script_skills_path is None:
    raise FileNotFoundError(f"无法找到 Skills/script.md，已搜索路径：{_possible_paths}，请确认 notebook 运行目录正确")

with open(script_skills_path, "r", encoding="utf-8") as f:
    script_skills_content = f.read()

print("✅ 已加载短视频剧本规则")
print(f"规则文件路径：{script_skills_path}")
print(f"规则总长度：{len(script_skills_content)} 字符")
print("=" * 50)
print("【规则摘要】")
print("- 黄金3秒法则：直接从冲突高潮切入")
print("- 压缩节拍：30-60秒完成起承转合")
print("- 高冲击视觉：特写、夸张动作描写")
print("- 视听语言：包含景别、运镜、BGM、音效")
print("- 钩子结尾：情绪最高点戛然而止")

### (3) 加载示例小说

In [ ]:
# 读取示例小说内容
# 使用与规则文件相同的路径查找逻辑
_novel_possible_paths = [
    os.path.join(_project_root, "sample_novel", "禁欲老公会读心，大馋丫头藏不住啦", "第一章 读心"),
    os.path.join(_cwd, "sample_novel", "禁欲老公会读心，大馋丫头藏不住啦", "第一章 读心"),
    os.path.join(_cwd, "..", "sample_novel", "禁欲老公会读心，大馋丫头藏不住啦", "第一章 读心"),
]

novel_path = None
for _np in _novel_possible_paths:
    _np_normalized = os.path.normpath(_np)
    if os.path.exists(_np_normalized):
        novel_path = os.path.abspath(_np_normalized)
        break

if novel_path is None:
    raise FileNotFoundError(f"找不到小说文件，已搜索路径：{_novel_possible_paths}")

with open(novel_path, "r", encoding="utf-8") as f:
    novel_content = f.read()

print("✅ 已加载示例小说")
print(f"小说文件路径：{novel_path}")
print(f"小说长度：{len(novel_content)} 字符")
print("=" * 50)
# 显示小说前200字符预览
print("【小说预览-前200字】")
print(novel_content[:200])

### (4) 使用 requests 调用 Agnes AI 生成短视频剧本

In [ ]:
import requests
import json

url = "https://apihub.agnes-ai.com/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# 构建系统提示词：结合短视频剧本规则
system_prompt = f"""{script_skills_content}

请严格按照上述规则，将用户提供的小说片段转换为高完播率的短视频拍摄脚本。
注意：
1. 脚本必须包含【场景编号】、【画面】、【运镜】、【台词】、【音效/BGM】等元素
2. 使用黄金3秒法则，直接从冲突最激烈的情节切入
3. 节奏紧凑，30-60秒内完成起承转合
4. 结尾设置钩子，引导观众点赞或看下一集
5. 输出格式严格按照规则中定义的格式
"""

user_prompt = f"""请根据以下小说内容，生成一个完整的短视频拍摄脚本：

---
{novel_content}
---

请按照短视频脚本格式输出，包含多个场景，每个场景包含画面、运镜、台词和音效/BGM。"""

payload = {
    "model": "agnes-2.0-flash",
    "messages": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    "temperature": 0.7,
    "max_tokens": 4096
}

print("🎬 正在调用 Agnes AI 生成短视频剧本...")
print("=" * 50)

response = requests.post(url, headers=headers, data=json.dumps(payload))
result = response.json()

# 输出生成的剧本
script_content = result["choices"][0]["message"]["content"]
print(script_content)

### (5) 将生成的剧本保存到文件

In [ ]:
# 保存生成的剧本为文本文件
output_path = "test/generated_script.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(script_content)

print(f"✅ 剧本已保存到: {output_path}")

### (6) 批量处理：将小说按章节分割并生成多个剧本

In [ ]:
# 如果小说有多章/多段，可以分段生成
# 这里演示将当前小说按自然段落分割后，选取前N段进行剧本生成

def split_novel_into_segments(text, max_chars=1000):
    """将小说按段落分割为多个片段，每个片段不超过 max_chars 字符"""
    paragraphs = text.split("\n\n")
    segments = []
    current = ""
    for para in paragraphs:
        if len(current) + len(para) < max_chars:
            current += para + "\n\n"
        else:
            if current:
                segments.append(current.strip())
            current = para + "\n\n"
    if current:
        segments.append(current.strip())
    return segments

segments = split_novel_into_segments(novel_content, max_chars=1200)
print(f"小说已分割为 {len(segments)} 个片段")
for i, seg in enumerate(segments):
    print(f"  片段 {i+1}: {len(seg)} 字符 - 预览: {seg[:60]}...")

### (7) 分段批量生成剧本（多段并行/串行）

In [ ]:
# 分段调用 API 生成剧本
# 注意：如果 AGNES_API_KEY 未设置，此单元格会跳过执行

if not api_key:
    print("⚠️ 请先设置 AGNES_API_KEY 环境变量")
else:
    # 只处理前3个片段用于演示
    max_segments = min(3, len(segments))
    
    for i in range(max_segments):
        print(f"\n{'='*60}")
        print(f"🎬 正在生成片段 {i+1}/{max_segments} 的剧本...")
        print(f"{'='*60}")
        
        segment_prompt = f"""请根据以下小说片段，生成一个完整的短视频拍摄脚本：

---
{segments[i]}
---

请按照短视频脚本格式输出，包含多个场景，每个场景包含画面、运镜、台词和音效/BGM。"""
        
        payload = {
            "model": "agnes-2.0-flash",
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": segment_prompt}
            ],
            "temperature": 0.8,
            "max_tokens": 2048
        }
        
        response = requests.post(url, headers=headers, data=json.dumps(payload))
        result = response.json()
        script_segment = result["choices"][0]["message"]["content"]
        
        print(f"\n【片段 {i+1} 生成的剧本】")
        print(script_segment)
        
        # 保存每个片段的剧本
        seg_output = f"test/script_segment_{i+1}.txt"
        with open(seg_output, "w", encoding="utf-8") as f:
            f.write(script_segment)
        print(f"✅ 已保存: {seg_output}")